In [1]:
import sys, os, random, hashlib, json
from pathlib import Path
import numpy as np
import pandas as pd
import ast

import wfdb
from scipy import signal as sp_signal
from scipy.signal import sosfiltfilt, iirnotch, tf2sos
from sklearn.metrics import roc_auc_score, precision_recall_curve

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.regularizers import l2

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [2]:
DATA_DIR  = Path(r"C:\Users\Usuario\Desktop\TFG\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3")
CACHE_DIR = DATA_DIR / "cache_v2" / "con_filtrado"
MODEL_DIR = Path("./outputs/models"); MODEL_DIR.mkdir(parents=True, exist_ok=True)

FS = 100
SUPER_CLASSES = ["NORM", "MI", "STTC", "HYP", "CD"]
CLASS_MAP_HUMAN = {"NORM": "Normal", "MI": "Infarto", "STTC": "Isquemia/ST-T",
                    "HYP": "Hipertrofia", "CD": "Conducción (bloqueos)"}

BANDPASS_LOW, BANDPASS_HIGH = 0.5, 40.0
NOTCH_FREQ, NOTCH_Q = 50.0, 30.0
STFT_NPERSEG, STFT_NOVERLAP = 64, 48
SPEC_SHAPE = (33, 64)
L2_REG = 1e-4
GAMMA = 1.0

CV_RESULTS_CSV = MODEL_DIR / "cv_results.csv"

In [3]:
df  = pd.read_csv(DATA_DIR / "ptbxl_database.csv", index_col="ecg_id")
scp = pd.read_csv(DATA_DIR / "scp_statements.csv", index_col=0)
df.columns  = df.columns.str.strip()
scp.columns = scp.columns.str.strip()

scp_super = (scp[scp["diagnostic_class"].notna()][["diagnostic_class"]]
             .rename(columns={"diagnostic_class": "superclass"}))

def extract_superclasses(scp_codes_str):
    if pd.isna(scp_codes_str):
        return []
    try:
        d = ast.literal_eval(scp_codes_str)
        if not isinstance(d, dict):
            return []
    except Exception:
        return []
    return list({scp_super.loc[k, "superclass"] for k in d.keys() if k in scp_super.index})

def multilabel_vector(classes):
    s = set(classes)
    return np.array([1 if c in s else 0 for c in SUPER_CLASSES], dtype=np.int32)

df["superclasses"] = df["scp_codes"].apply(extract_superclasses)
df = df[df["superclasses"].apply(len) > 0].reset_index()
df["target_vec"] = df["superclasses"].apply(multilabel_vector)

In [4]:
def is_valid_signal(x, flat_std_thresh=1e-6):
    if np.any(np.isnan(x)) or np.any(np.isinf(x)):
        return False
    return np.std(x) >= flat_std_thresh

_BP_SOS    = sp_signal.butter(4, [BANDPASS_LOW / (FS/2), BANDPASS_HIGH / (FS/2)], btype="band", output="sos")
_NOTCH_SOS = tf2sos(*iirnotch(NOTCH_FREQ / (FS/2), NOTCH_Q))

def clean_lead(x):
    x = np.asarray(x, dtype=np.float64)
    if not is_valid_signal(x):
        return None
    x = sosfiltfilt(_BP_SOS, x)
    x = sosfiltfilt(_NOTCH_SOS, x)
    mu, sigma = x.mean(), x.std()
    if sigma < 1e-8:
        return None
    return ((x - mu) / sigma).astype(np.float32)

def preprocess_ecg(raw_signal):
    cleaned = np.zeros_like(raw_signal, dtype=np.float32)
    n_bad = 0
    for i in range(raw_signal.shape[1]):
        c = clean_lead(raw_signal[:, i])
        cleaned[:, i] = c if c is not None else 0.0
        n_bad += (c is None)
    return None if n_bad > raw_signal.shape[1] // 2 else cleaned

def lead_to_spectrogram(lead_signal, fs=FS, nperseg=STFT_NPERSEG, noverlap=STFT_NOVERLAP, out_shape=SPEC_SHAPE):
    _, _, Zxx = sp_signal.stft(lead_signal, fs=fs, nperseg=nperseg, noverlap=noverlap)
    mag = np.log1p(np.abs(Zxx))
    F, T = out_shape
    mag = mag[:F, :T]
    if mag.shape != (F, T):
        pad = np.zeros((F, T), dtype=np.float32)
        pad[:mag.shape[0], :mag.shape[1]] = mag
        mag = pad
    mn, mx = mag.min(), mag.max()
    if mx - mn > 1e-8:
        mag = (mag - mn) / (mx - mn)
    return mag.astype(np.float32)

def ecg_to_tensor(signal_2d):
    specs = [lead_to_spectrogram(signal_2d[:, i]) for i in range(signal_2d.shape[1])]
    return np.stack(specs, axis=-1)

def _cache_path(rel_path):
    h = hashlib.md5(rel_path.encode()).hexdigest()
    return CACHE_DIR / f"{h}.npy"

def get_tensor_cached(rel_path):
    cp = _cache_path(rel_path)
    if cp.exists():
        return np.load(cp)
    raw, _ = wfdb.rdsamp(str(DATA_DIR / rel_path))
    clean = preprocess_ecg(raw)
    if clean is None:
        return None
    tensor = ecg_to_tensor(clean)
    np.save(cp, tensor)
    return tensor

def build_split(df_split):
    Xs, Ys = [], []
    for _, row in df_split.iterrows():
        t = get_tensor_cached(row["filename_lr"])
        if t is None:
            continue
        Xs.append(t)
        Ys.append(row["target_vec"])
    return np.stack(Xs).astype(np.float32), np.stack(Ys).astype(np.float32)

In [5]:
def build_model_deep(input_shape, n_classes):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, 3, padding="same", activation="relu", kernel_regularizer=l2(L2_REG))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu", kernel_regularizer=l2(L2_REG))(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(128, 3, padding="same", activation="relu", kernel_regularizer=l2(L2_REG))(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Dense(64, activation="relu", kernel_regularizer=l2(L2_REG))(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(n_classes, activation="sigmoid")(x)
    return models.Model(inputs, outputs)

def make_focal_weighted_bce(pos_weight_tf, gamma=GAMMA):
    def focal_weighted_bce(y_true, y_pred):
        eps = 1e-7
        y_pred = tf.clip_by_value(y_pred, eps, 1 - eps)
        pos_term = pos_weight_tf * y_true * tf.pow(1 - y_pred, gamma) * tf.math.log(y_pred)
        neg_term = (1 - y_true) * tf.pow(y_pred, gamma) * tf.math.log(1 - y_pred)
        return tf.reduce_mean(-(pos_term + neg_term))
    return focal_weighted_bce

In [6]:
FOLD_GROUPS = [(1,2), (3,4), (5,6), (7,8), (9,10)]

def evaluate_fold(test_folds, fold_id):
    remaining = [f for f in range(1, 11) if f not in test_folds]
    val_folds = remaining[:2]
    train_folds = remaining[2:]

    train_df = df[df["strat_fold"].isin(train_folds)].reset_index(drop=True)
    val_df   = df[df["strat_fold"].isin(val_folds)].reset_index(drop=True)
    test_df  = df[df["strat_fold"].isin(test_folds)].reset_index(drop=True)

    X_train, Y_train = build_split(train_df)
    X_val,   Y_val   = build_split(val_df)
    X_test,  Y_test  = build_split(test_df)

    pos_counts = Y_train.sum(axis=0)
    neg_counts = len(Y_train) - pos_counts
    pos_weight = np.clip(neg_counts / np.maximum(pos_counts, 1), 1.0, 10.0)
    pos_weight_tf = tf.constant(pos_weight, dtype=tf.float32)

    model = build_model_deep(X_train.shape[1:], len(SUPER_CLASSES))
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                   loss=make_focal_weighted_bce(pos_weight_tf),
                   metrics=[tf.keras.metrics.AUC(name="auc", multi_label=True)])

    cbs = [
        callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=8, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", patience=4),
    ]
    model.fit(X_train, Y_train, validation_data=(X_val, Y_val),
              epochs=60, batch_size=32, callbacks=cbs, verbose=0)

    val_probs = model.predict(X_val, verbose=0)
    best_thresholds = {}
    for i, c in enumerate(SUPER_CLASSES):
        prec, rec, thr = precision_recall_curve(Y_val[:, i], val_probs[:, i])
        f1 = 2 * prec * rec / np.clip(prec + rec, 1e-8, None)
        best_i = np.nanargmax(f1[:-1]) if len(thr) > 0 else 0
        best_thresholds[c] = float(thr[best_i]) if len(thr) > 0 else 0.5

    test_probs = model.predict(X_test, verbose=0)
    auc_por_clase = {}
    for i, c in enumerate(SUPER_CLASSES):
        auc_por_clase[c] = np.nan if len(np.unique(Y_test[:, i])) < 2 else roc_auc_score(Y_test[:, i], test_probs[:, i])
    auc_macro = float(np.nanmean(list(auc_por_clase.values())))

    return {"fold_id": fold_id, "test_folds": str(test_folds), "auc_macro": auc_macro,
            **{f"auc_{c}": auc_por_clase[c] for c in SUPER_CLASSES}}

In [7]:
if CV_RESULTS_CSV.exists():
    cv_results = pd.read_csv(CV_RESULTS_CSV)
    completed = set(cv_results["fold_id"])
else:
    cv_results = pd.DataFrame()
    completed = set()

for i, test_folds in enumerate(FOLD_GROUPS):
    if i in completed:
        print(f"Fold {i} ya calculado, se omite.")
        continue
    print(f"Calculando fold {i} (test={test_folds})...")
    result = evaluate_fold(test_folds, i)
    cv_results = pd.concat([cv_results, pd.DataFrame([result])], ignore_index=True)
    cv_results.to_csv(CV_RESULTS_CSV, index=False)
    print(result)

Calculando fold 0 (test=(1, 2))...
{'fold_id': 0, 'test_folds': '(1, 2)', 'auc_macro': 0.88942715388294, 'auc_NORM': 0.9349861381231482, 'auc_MI': 0.8668155718912309, 'auc_STTC': 0.9239201971461034, 'auc_HYP': 0.8206570524984187, 'auc_CD': 0.9007568097557987}
Calculando fold 1 (test=(3, 4))...
{'fold_id': 1, 'test_folds': '(3, 4)', 'auc_macro': 0.8950827091343513, 'auc_NORM': 0.9365068573582948, 'auc_MI': 0.879317717847093, 'auc_STTC': 0.9128594106217077, 'auc_HYP': 0.8464406822649166, 'auc_CD': 0.9002888775797442}
Calculando fold 2 (test=(5, 6))...
{'fold_id': 2, 'test_folds': '(5, 6)', 'auc_macro': 0.8952834071243089, 'auc_NORM': 0.9405308678795108, 'auc_MI': 0.8826870553254752, 'auc_STTC': 0.9123121451958636, 'auc_HYP': 0.8308248308909959, 'auc_CD': 0.910062136329699}
Calculando fold 3 (test=(7, 8))...
{'fold_id': 3, 'test_folds': '(7, 8)', 'auc_macro': 0.8953894108716408, 'auc_NORM': 0.9359440629828116, 'auc_MI': 0.8762612899490504, 'auc_STTC': 0.9182032032002051, 'auc_HYP': 0.8364

In [8]:
resumen = cv_results.drop(columns=["fold_id", "test_folds"]).agg(["mean", "std"])
print(resumen)

      auc_macro  auc_NORM    auc_MI  auc_STTC   auc_HYP    auc_CD
mean   0.891241  0.934512  0.873454  0.917348  0.828888  0.902001
std    0.006246  0.005935  0.008638  0.004843  0.014058  0.008793
